In [22]:
def print_obj_lines(obj, *, title=None, fields=None, content_preview=180):
    if title:
        print(f"\n=== {title} ===")

    if hasattr(obj, "model_dump"):
        data = obj.model_dump(mode="json")
    elif hasattr(obj, "__dict__"):
        data = {k: v for k, v in vars(obj).items() if not k.startswith("_")}
    elif isinstance(obj, dict):
        data = obj
    else:
        data = {"value": str(obj)}

    if fields is None:
        items = data.items()
    else:
        items = ((k, data.get(k)) for k in fields)

    for k, v in items:
        # Avoid dumping huge episode content in one line
        if k == "content" and isinstance(v, str):
            preview = v[:content_preview]
            print(f"{k}_preview: {preview}{'...<truncated>' if len(v) > content_preview else ''}")
            print(f"{k}_length: {len(v)}")
        else:
            print(f"{k}: {v}")


def print_graph_search_results(results, *, show_top_level=True, content_preview=180):
    if show_top_level:
        print("\n===== GraphSearchResults Summary =====")
        print(f"communities: {getattr(results, 'communities', None)}")
        print(f"context: {getattr(results, 'context', None)}")
        print(f"sagas: {getattr(results, 'sagas', None)}")
        print(f"themes: {getattr(results, 'themes', None)}")

    edges = getattr(results, "edges", []) or []
    nodes = getattr(results, "nodes", []) or []
    episodes = getattr(results, "episodes", []) or []

    print(f"edges: {len(edges)}")
    print(f"nodes: {len(nodes)}")
    print(f"episodes: {len(episodes)}")

    for i, edge in enumerate(edges, 1):
        print_obj_lines(
            edge,
            title=f"Edge #{i}",
            fields=[
                "uuid_", "name", "fact", "score", "relevance", "scope",
                "source_node_uuid", "target_node_uuid",
                "created_at", "valid_at", "expired_at", "invalid_at",
                "graph_id", "attributes", "episodes",
            ],
            content_preview=content_preview,
        )

    for i, node in enumerate(nodes, 1):
        print_obj_lines(
            node,
            title=f"Node #{i}",
            fields=[
                "uuid_", "name", "summary", "labels", "score", "relevance",
                "created_at", "graph_id", "attributes",
            ],
            content_preview=content_preview,
        )

    for i, episode in enumerate(episodes, 1):
        print_obj_lines(
            episode,
            title=f"Episode #{i}",
            fields=[
                "uuid_", "score", "relevance", "source",
                "created_at", "thread_id", "session_id",
                "role", "role_type", "processed", "metadata",
                "content",
            ],
            content_preview=content_preview,
        )

In [23]:
from pathlib import Path

from dotenv import load_dotenv

# Load .env from project root (same directory as this notebook when opened from repo root)
_env = Path.cwd() / ".env"
if not _env.is_file():
    _env = Path.cwd().parent / ".env"

print(_env)
load_dotenv(_env)


import os

ZEP_API_KEY = os.environ["ZEP_API_KEY"]
from zep_cloud import Zep
client = Zep(api_key=ZEP_API_KEY)
print(ZEP_API_KEY)



/Users/freedomkwokmacbookpro/Github/imp/imp_agent_core/.env
z_1dWlkIjoiNmUyZTlmMjctNzhhMS00YmU0LTlkNTYtMjQ3NTQzZGQyYmNmIn0.F9YyvzyCmYQRz06MZ1JstNE_v7BghkXMtzd91lGfgCvrR431q576XVzJUFlyQ-poe15R4gWUKXeYNpxs220Wkg


In [27]:
node_hits = client.graph.node.get_by_graph_id(
    graph_id='proj_63edb3c4f72f',
    limit=10,
)
node_hits

[EntityNode(attributes={'name': '短信'}, created_at='2026-04-09T06:54:19.096Z', labels=[], name='短信', relevance=None, score=None, selection_rank=None, summary='该平台适用于刚认识阶段对话分析与破冰的场景。', uuid_='fceb7f66-f926-4ac3-bdde-e3eceb48f2cf', graph_id=''),
 EntityNode(attributes={'name': '刚认识阶段对话分析与破冰'}, created_at='2026-04-09T06:54:19.096Z', labels=[], name='刚认识阶段对话分析与破冰', relevance=None, score=None, selection_rank=None, summary='该技能目标是帮助用户分析刚认识阶段的聊天记录或拟发送消息，以判断互动节奏、识别社交动作、判断推进速度是否过快或任务化，并找到当前卡点。它提供低风险、自然的下一步回复，以在保留体面的同时继续推进。该技能适用于刚打招呼、刚认识、刚匹配、刚加好友初期，以及在 dating app、微信、Instagram、短信、社交平台私信等场景中，聊天未建立稳定熟悉感或用户想推进但怕太快的情况。', uuid_='e3d08294-e50f-4bf9-a00b-1e3bdc3ffa5f', graph_id=''),
 EntityNode(attributes={'name': '点卡'}, created_at='2026-04-09T06:54:42.107Z', labels=[], name='点卡', relevance=None, score=None, selection_rank=None, summary="The information provided concerns guidelines on inappropriate use cases, stating that it does not apply to judging the suitability of one's own statement or determining 

In [32]:
node_hits[0]

EntityNode(attributes={'name': '短信'}, created_at='2026-04-09T06:54:19.096Z', labels=[], name='短信', relevance=None, score=None, selection_rank=None, summary='该平台适用于刚认识阶段对话分析与破冰的场景。', uuid_='fceb7f66-f926-4ac3-bdde-e3eceb48f2cf', graph_id='')

In [31]:
edges = client.graph.node.get_edges(node_uuid="e3d08294-e50f-4bf9-a00b-1e3bdc3ffa5f")
edges

[]

In [ ]:
# --- Node retrieval examples ---
# Use your Zep user id (or graph_id in search where applicable).
graph_id = "proj_63edb3c4f72f"
NODE_UUID = "replace-with-node-uuid"  # from search results or dashboard

# 1) Hybrid search scoped to nodes
node_hits = client.graph.search(
    query='skill',
    graph_id='proj_63edb3c4f72f',
    scope='nodes',
    reranker="mmr",
    limit=10,
)



print_graph_search_results(node_hits)
# 
# node = node_hits.nodes[0]
# print(node.name)
# print(node.attributes)
# print(node.labels)
# print(node.summary)
# print(node)




===== GraphSearchResults Summary =====
communities: None
context: None
sagas: None
themes: None
edges: 0
nodes: 9
episodes: 0

=== Node #1 ===
uuid_: 73f13670-e03f-4bb3-b04c-dfd3a6cb6321
name: Instagram
summary: 该平台适用于刚认识阶段对话分析与破冰的场景。
labels: []
score: 0.016393442
relevance: None
created_at: 2026-04-09T06:54:19.096Z
graph_id: 
attributes: {'name': 'Instagram'}

=== Node #2 ===
uuid_: e3d08294-e50f-4bf9-a00b-1e3bdc3ffa5f
name: 刚认识阶段对话分析与破冰
summary: 该技能目标是帮助用户分析刚认识阶段的聊天记录或拟发送消息，以判断互动节奏、识别社交动作、判断推进速度是否过快或任务化，并找到当前卡点。它提供低风险、自然的下一步回复，以在保留体面的同时继续推进。该技能适用于刚打招呼、刚认识、刚匹配、刚加好友初期，以及在 dating app、微信、Instagram、短信、社交平台私信等场景中，聊天未建立稳定熟悉感或用户想推进但怕太快的情况。
labels: []
score: 0.014925373
relevance: None
created_at: 2026-04-09T06:54:19.096Z
graph_id: 
attributes: {'name': '刚认识阶段对话分析与破冰'}

=== Node #3 ===
uuid_: bfad2a7b-7e62-4f7a-8145-db29942fbd70
name: 点卡
summary: The information provided concerns guidelines on inappropriate use cases, stating that it does not apply to judging the suitability of one's own sta

In [17]:
node_hits

GraphSearchResults(communities=None, context=None, edges=[], episodes=[Episode(content='=== init_start.md ===\n# Skill: 刚认识阶段对话分析与破冰\n\n## Skill 名称\n\n刚认识对话分析与破冰\n\n## Skill 目标\n\n当用户提供刚认识阶段的聊天截图、聊天记录、拟发送消息，或询问“这样回可以吗 / 对方什么意思 / 怎么破冰 / 要不要推进微信或见面”时，帮助用户：\n\n* 判断当前互动处于什么节奏\n* 识别自己上一句属于哪种社交动作\n* 判断推进是否过快、过直、过任务化\n* 找到当前真正的卡点\n* 给出低风险、自然、不尴尬的下一步回复\n* 在保留体面和后续空间的前提下继续推进\n\n---\n\n## 适用范围\n\n适用于以下场景：\n\n* 刚打招呼，刚认识、刚匹配、刚加好友初期\n* dating app、微信、Instagram、短信、社交平台私信\n* 聊天还未建立稳定熟悉感\n* 用户想推进但怕太快\n* 对方没接微信、没接邀约、回得一般、节奏有点卡\n* 用户想判断自己这句是否合适，以及下一句该怎么发\n\n---\n\n## 不适用范围\n\n不适用于：', created_at='2026-04-09T06:54:12.659536Z', metadata=None, processed=True, relevance=None, role=None, role_type=None, score=2.2100365, selection_rank=None, source='text', source_description='', task_id=None, thread_id=None, uuid_='8e94185c-53f0-4f1e-ac61-e3d782b40048', session_id=None)], nodes=[], sagas=None, themes=None)

In [11]:
node_hits.nodes[0]

EntityNode(attributes={'name': 'Instagram'}, created_at='2026-04-09T06:54:19.096Z', labels=[], name='Instagram', relevance=None, score=0.016393442, summary='该平台适用于刚认识阶段对话分析与破冰的场景。', uuid_='73f13670-e03f-4bb3-b04c-dfd3a6cb6321', graph_id='')

In [ ]:

# 2) Single node by UUID
node = client.graph.node.get(uuid_=NODE_UUID)
print("node by uuid:", node)

# 3) All nodes for a user (paginate with uuid_cursor if needed)
nodes_page = client.graph.node.get_by_user_id(USER_ID, limit=50)
print("nodes for user (first page):", len(nodes_page), "items")

In [ ]:
# --- Edge retrieval examples ---

USER_ID = "proj_63edb3c4f72f"
EDGE_UUID = "replace-with-edge-uuid"
NODE_UUID = "replace-with-node-uuid"

# 1) Hybrid search scoped to edges (facts / relationships)
node_hits = client.graph.search(
    query="best skill for conversation",
    graph_id=USER_ID,
    scope="nodes",
    limit=10,
)

print("node from search:", node_hits.nodes)

# 2) Single edge by UUID
edge = client.graph.edge.get(uuid_=EDGE_UUID)
print("edge by uuid:", edge)

# 3) Edges linked to a user’s graph
edges_page = client.graph.edge.get_by_user_id(USER_ID, limit=50)
print("edges for user (first page):", len(edges_page), "items")

# 4) Entity edges attached to a specific node
entity_edges = client.graph.node.get_edges(node_uuid=NODE_UUID)
print("entity edges for node:", entity_edges)
